## Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoModel, AutoTokenizer
import os 
import sys
from pathlib import Path


In [ ]:
# function that calculates single vector that averages meaning of entire sentence

def average_pool(
    last_hidden_states: Tensor,
    attention_mask: Tensor,
):
    last_hidden = last_hidden_states.masked_fill(
        ~attention_mask[..., None].bool(), 0.0
    )
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]



#### Configuration

In [ ]:
ROOT = Path(os.path.abspath('')).parent.parent
print(ROOT)
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
PROCESSED_FOLDER = ROOT / 'data/processed'
EMBEDDINGS_FOLDER = ROOT / 'data/embeddings'
EMBEDDINGS_FILE = EMBEDDINGS_FOLDER / 'open_source_embeddings.npy'
METADATA_FILE = EMBEDDINGS_FOLDER / 'metadata.jsonl'
CHUNKS_FILE = PROCESSED_FOLDER / 'all_chunks.jsonl'
BATCH_SIZE = 32

### Load data

In [ ]:
chunks_df = pd.read_json(CHUNKS_FILE, lines=True)
chunks_df.head()

In [ ]:
text_to_embed = chunks_df['page_content'].tolist()
print(f'Total chunks to embed: {len(text_to_embed)}')

### Load model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)

### Generate embeddings

In [ ]:
from tqdm import tqdm

all_embeddings = []
print('Generating embeddings with batch size:', BATCH_SIZE)

for i in tqdm(range(0, len(text_to_embed), BATCH_SIZE)):
    batch_texts = text_to_embed[i:i+BATCH_SIZE]
    # tokenize per batch
    encoded_input = tokenizer(
        batch_texts, 
        padding=True,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # pool and normalize embeddings
    batch_embeddings = average_pool(model_output.last_hidden_state, encoded_input['attention_mask'])
    batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)

    all_embeddings.extend(batch_embeddings.cpu().numpy())


In [ ]:
embeddings_np = np.array(all_embeddings)
print('Embeddings shape:', embeddings_np.shape)

### Save results

In [ ]:
np.save(EMBEDDINGS_FILE, embeddings_np)
chunks_df['metadata'].to_json(METADATA_FILE, orient='records', lines=True)
